# 04 · 자연어 기반 취합 → 위키 Markdown 생성

**파이프라인 순서 4/4** — 자연어 질의 한 줄로 두 MCP에서 관련 데이터만 취합하고,
LLM Wiki용 Markdown 초안을 생성합니다. 검토·수정 후 저장/커밋하면 `app/wiki/`에 반영됩니다.

이 노트북은 웹앱(`app/main.py`)이 내부적으로 호출하는 `run_pipeline`을 그대로 사용합니다.

In [ ]:
# --- Bootstrap: locate llmwiki-pipeline/app and import the shared package ---
import sys
from pathlib import Path

def _find_app_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'app', base / 'llmwiki-pipeline' / 'app'):
            if (cand / 'pipeline' / '__init__.py').is_file():
                return cand
    raise RuntimeError('Could not locate llmwiki-pipeline/app — run this from inside the project.')

APP_DIR = _find_app_dir()
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
print('app dir on sys.path:', APP_DIR)

## 1. 무엇을, 언제 범위로 추출할지 지정

`QUERY`에 추출 주제를 자연어로 적고, 날짜 범위를 지정합니다(기본: 최근 1일).

In [ ]:
from pipeline import extract

QUERY = 'Postgres 튜닝, Kubernetes 트러블슈팅, CI 캐시 등 재사용 가능한 기술 노하우'

# 날짜 범위: 기본은 최근 1일. 넓히려면 days 값을 늘리세요.
START, END = extract.default_date_range(days=7)
SOURCES = None   # None이면 전체(mail+teams). 특정 소스만: ['mail'] 또는 ['teams']
print('질의:', QUERY)
print('범위:', START, '..', END)

## 2. 파이프라인 실행 (extract → generate)

MCP에서 관련 자료를 취합한 뒤 LLM이 Markdown 초안을 만듭니다. **저장/커밋은 하지 않습니다.**

In [ ]:
from pipeline import pipeline

result = await pipeline.run_pipeline(QUERY, start=START, end=END, sources=SOURCES)

if result.get('token_errors'):
    print('토큰 경고:', result['token_errors'])
if result.get('source_errors'):
    print('소스 경고:', result['source_errors'])

doc = result['doc']
assert doc, '초안이 생성되지 않았습니다. 로그인/LLM 설정과 위 경고를 확인하세요.'
print('제목:', doc['title'])
print('slug:', doc['slug'])

## 3. 초안 미리보기

In [ ]:
from IPython.display import Markdown, display
markdown_text = doc['markdown']
display(Markdown(markdown_text))

## 4. 검토·수정

아래 `markdown_text`를 직접 편집하거나, 위 미리보기를 보고 문자열을 수정하세요.
(웹앱에서는 이 단계를 편집 UI로 수행합니다.)

In [ ]:
# 예: 특정 문구 치환 / 섹션 추가 등 자유롭게 수정
# markdown_text = markdown_text.replace('오타', '수정')
print(markdown_text[:800])

## 5. 저장 + 커밋 → `app/wiki/`

검토가 끝나면 저장하고 커밋합니다. 커밋은 **해당 문서 파일만** 포함합니다(pathspec 제한).
저장만 하고 커밋은 나중에 하려면 `wiki.save_doc(...)`을 사용하세요.

In [ ]:
from pipeline import wiki

DO_COMMIT = False   # 실제로 커밋하려면 True 로 바꾸세요

if DO_COMMIT:
    out = wiki.save_and_commit(markdown_text, doc['slug'])
    print('저장:', out['path'])
    print('커밋:', out['commit'])
else:
    path = wiki.save_doc(markdown_text, doc['slug'])
    print('저장만 완료(커밋 안 함):', path)
    print('나중에 커밋하려면 DO_COMMIT=True 로 다시 실행하세요.')

## 6. 저장된 위키 문서 목록

In [ ]:
from pipeline import wiki
for d in wiki.list_docs():
    print(f"- {d['filename']}  |  {d.get('title','')}")

🎉 완료! 생성된 Markdown이 `app/wiki/`에 저장/커밋되어 LLM Wiki에 반영됩니다.
동일한 흐름을 웹 UI로 쓰려면 프로젝트 루트에서 `uvicorn app.main:app --reload` 후 브라우저로 접속하세요.